# 🛴 데이터 모델링: 정규화(Normalization) 시연
### 공유 킥보드 대여 서비스 DB를 예시로 사용합니다.

---

## 이 노트북에서 배우는 것

| 단계 | 내용 | 핵심 질문 |
|------|------|-----------|
| Step 0 | 비정규화 테이블 생성 | 왜 처음부터 한 테이블에 다 넣으면 안 될까? |
| Step 1 | 이상 현상(Anomaly) 확인 | 잘못된 설계는 INSERT/UPDATE/DELETE에서 무엇이 깨지나? |
| Step 2 | 1차 정규화 (1NF) | **원자값(Atomic Value)** 을 어떻게 보장하나? |
| Step 3 | 2차 정규화 (2NF) | **부분 함수 종속**을 어떻게 제거하나? |
| Step 4 | 3차 정규화 (3NF) | **이행 함수 종속**을 어떻게 제거하나? |
| Step 5 | 최종 결과 검증 | 정규화 후에도 JOIN으로 원본과 같은 정보를 볼 수 있나? |

---

## 배경: 정규화란?

**정규화(Normalization)** 는 관계형 데이터베이스에서 **중복·종속 관계를 분석해 테이블을 분해**하고,  
**삽입·갱신·삭제 이상(Anomaly)** 이 발생하지 않도록 스키마를 개선하는 과정입니다.

- 이론적 토대: **함수 종속(Functional Dependency)** — 한 속성 집합이 다른 속성을 유일하게 결정하는 관계
- 실무 목표: **무결성(Integrity)** 유지 + **중복 최소화** + **유지보수 비용 감소**
- 트레이드오프: 테이블이 늘어나면 **JOIN이 많아져** 조회가 복잡해질 수 있음 → 필요 시 **역정규화(Denormalization)** 로 균형

> 이 데모는 교육용으로 **3NF까지** 진행합니다. BCNF, 4NF, 5NF는 고급 주제로 별도 학습합니다.

---

## 도메인: 공유 킥보드 대여

| 엔티티 | 예시 속성 | 비고 |
|--------|-----------|------|
| 고객(Customer) | customer_id, customer_name | 회원 계정 |
| 대여(Rental) | rental_time, rental_location | 언제·어디서 빌렸는지 |
| 킥보드(Kickboard) | brand, model_year, price_per_min, basic_price, company | 기기·요금·제조사 |

비정규화 상태에서는 위 세 엔티티가 **한 행(row)** 에 섞여 있습니다.

In [9]:
import sqlite3
import pandas as pd
from IPython.display import display

# DB 연결 (파일로 저장해서 단계 간 상태 유지)
conn = sqlite3.connect("demo.db")
conn.execute("PRAGMA foreign_keys = ON")


def drop_from_1nf() -> None:
    """1NF부터 다시 만들 때 — 1NF·2NF·3NF 테이블 모두 삭제 (자식 → 부모 순)"""
    conn.executescript("""
        DROP TABLE IF EXISTS rental_3nf;
        DROP TABLE IF EXISTS kickboard_3nf;
        DROP TABLE IF EXISTS rental_2nf;
        DROP TABLE IF EXISTS customer_2nf;
        DROP TABLE IF EXISTS customer_1nf;
    """)
    conn.commit()


def drop_from_2nf() -> None:
    """2NF부터 다시 만들 때 — 3NF·2NF만 삭제 (customer_1nf는 유지)"""
    conn.executescript("""
        DROP TABLE IF EXISTS rental_3nf;
        DROP TABLE IF EXISTS kickboard_3nf;
        DROP TABLE IF EXISTS rental_2nf;
        DROP TABLE IF EXISTS customer_2nf;
    """)
    conn.commit()


def show(query, conn=conn, title=""):
    """SQL 결과를 pandas DataFrame으로 보기 좋게 출력"""
    if title:
        print(f"📋 {title}")
    df = pd.read_sql_query(query, conn)
    display(df)
    return df

## Step 0: 비정규화(Denormalized) 테이블

모든 정보를 **하나의 테이블(`customer_unnormalized`)** 에 담은 상태입니다.  
고객 정보, 대여 이력, 킥보드·요금·제조사 정보가 **한 행에 뒤섞여** 있습니다.

### 테이블 구조와 설계 의도(의도적 결함)

| 컬럼 | 의미 | 설계상 문제 |
|------|------|-------------|
| `customer_id`, `customer_name` | 고객 식별·이름 | 고객은 대여 없이도 존재할 수 있어야 함 |
| `rental_time` | 대여 시각 | **쉼표로 여러 값** → 1NF 위반(다중값) |
| `rental_location` | 대여 위치 | 대여 건마다 달라질 수 있음 |
| `brand`, `model_year`, `price_per_min`, `basic_price`, `company` | 킥보드·요금 | `brand`만 알면 나머지가 결정됨 → 중복 저장 |

### 비정규화 테이블의 문제점 (요약)

1. **의미 혼재(Mixed Granularity)**  
   한 행이 “고객 1명”인지 “대여 1건”인지 “고객+대여+기기” 묶음인지 불분명합니다.

2. **데이터 중복(Redundancy)**  
   `kmax6` 고객이 두 번 대여하면, `customer_name`, `brand`, `company` 등이 **행마다 반복** 저장됩니다.

3. **무결성 규칙 표현 불가**  
   “브랜드별 요금은 하나” 같은 규칙을 DB가 강제하기 어렵고, 애플리케이션에만 의존하게 됩니다.

4. **확장성 저하**  
   대여 횟수가 늘수록 같은 고객·같은 브랜드 정보가 **기하급수적으로 중복**됩니다.

5. **PRIMARY KEY 부재**  
   이 테이블에는 PK가 없어 **행을 유일하게 식별**하기 어렵습니다. (실무에서는 치명적)

### 향후 개선 포인트 (이 단계 이후 로드맵)

| 우선순위 | 개선 방향 | 기대 효과 |
|----------|-----------|-----------|
| 1 | `rental_time` 다중값 분리 → **1NF** | 컬럼당 하나의 값, 행 단위 의미 명확 |
| 2 | 고객 vs 대여 분리 → **2NF** | 삽입 이상 완화, 고객만 먼저 등록 가능 |
| 3 | 브랜드(킥보드) 마스터 분리 → **3NF** | 갱신 이상 제거, 요금 변경 시 한 곳만 수정 |
| 4 | PK·FK·NOT NULL 제약 추가 | DB 수준 무결성 보장 |
| 5 | (선택) `rental_id` 서로게이트 키 | 복합키 대신 조인·인덱스 단순화 |

아래 셀에서 실제 데이터를 생성하고, Step 1에서 **삽입·갱신·삭제 이상**을 코드로 확인합니다.

In [10]:
conn.executescript("""
DROP TABLE IF EXISTS customer_unnormalized;

CREATE TABLE customer_unnormalized (
    customer_id      TEXT,
    customer_name    TEXT,
    rental_time      TEXT,   -- 여러 시간이 쉼표로 합쳐질 수 있음
    rental_location  TEXT,
    brand            TEXT,
    model_year       INTEGER,
    price_per_min    INTEGER,
    basic_price      INTEGER,
    company          TEXT
);

INSERT INTO customer_unnormalized VALUES
    ('kmax6',     '김민준', '2020-08-20 13:01:02, 2020-09-01 20:39:52', '서울시 강남구 역삼동', 'boardkick', 2015, 100, 1000, 'elice'),
    ('flykite',   '이서연', '2020-11-11 22:01:30',                      '서울시 동작구 대방동', 'willgo',    2018, 110,  950, 'everythere'),
    ('freeman123','박서준', '2021-01-05 09:15:00',                      '서울시 관악구 신림동', 'boardkick', 2015, 100, 1000, 'elice');
""")
conn.commit()

show("SELECT * FROM customer_unnormalized", title="비정규화 테이블")

📋 비정규화 테이블


,customer_id,customer_name,rental_time,rental_location,brand,model_year,price_per_min,basic_price,company
0,kmax6,김민준,"2020-08-20 13:01:02, 2020-09-01 20:39:52",서울시 강남구 역삼동,boardkick,2015,100,1000,elice
1,flykite,이서연,2020-11-11 22:01:30,서울시 동작구 대방동,willgo,2018,110,950,everythere
2,freeman123,박서준,2021-01-05 09:15:00,서울시 관악구 신림동,boardkick,2015,100,1000,elice


,customer_id,customer_name,rental_time,rental_location,brand,model_year,price_per_min,basic_price,company
0,kmax6,김민준,"2020-08-20 13:01:02, 2020-09-01 20:39:52",서울시 강남구 역삼동,boardkick,2015,100,1000,elice
1,flykite,이서연,2020-11-11 22:01:30,서울시 동작구 대방동,willgo,2018,110,950,everythere
2,freeman123,박서준,2021-01-05 09:15:00,서울시 관악구 신림동,boardkick,2015,100,1000,elice


## Step 1: 이상 현상 (Anomaly)

잘못된 DB 설계는 데이터를 **삽입(INSERT)·갱신(UPDATE)·삭제(DELETE)** 할 때  
의도와 다른 부작용을 일으킵니다. 이를 **이상 현상**이라고 합니다.

> 📖 용어는 Codd(1970, 1972)의 관계형 모델과 이후 정규화 이론에서 체계화되었습니다.

---

### 1) 삽입 이상 (Insert Anomaly)

**정의:** 다른 정보 없이는 **필수 데이터를 넣을 수 없거나**, 넣어도 **의미가 불명확**해지는 현상.

| 이 데모에서 | 설명 |
|-------------|------|
| 증상 | 대여 이력 없는 신규 고객 `newuser`를 넣으면 `rental_time`, `brand` 등이 NULL |
| 원인 | 고객·대여·기기가 **한 테이블**에 묶여 있어 “대여 없는 고객” 행의 의미가 모호 |
| 실무 영향 | 회원가입 API와 대여 API가 **같은 테이블**을 건드려야 함 → 결합도↑ |
| **개선 포인트** | `customer` 테이블을 분리(→ **2NF**)하여 **대여 전 회원 등록** 허용 |

---

### 2) 갱신 이상 (Update Anomaly)

**정의:** 같은 사실이 **여러 행에 중복** 저장되어, 일부만 수정하면 **불일치(Inconsistency)** 가 생기는 현상.

| 이 데모에서 | 설명 |
|-------------|------|
| 증상 | `kmax6`의 이름을 바꿀 때 **한 행만** 수정하면 같은 ID인데 이름이 다름 |
| 원인 | `customer_name`이 대여 행마다 **반복 저장** (1NF 후에는 대여 2행마다 중복) |
| 실무 영향 | “고객명 변경”이 **N건의 대여 row**를 모두 갱신해야 함 → 누락·버그 위험 |
| **개선 포인트** | 고객명을 `customer` 테이블 **한 곳**에만 저장(→ **2NF**) |

---

### 3) 삭제 이상 (Delete Anomaly)

**정의:** 특정 정보를 지우려다 **함께 저장된 다른 정보까지** 의도치 않게 삭제되는 현상.

| 이 데모에서 | 설명 |
|-------------|------|
| 증상 | `flykite`의 **대여 기록**을 지우면 **이서연 고객 정보 전체** 소실 |
| 원인 | 고객 마스터와 대여 트랜잭션이 **같은 row**에 존재 |
| 실무 영향 | “대여 이력 보관 기간 만료 후 삭제” 시 **회원 계정까지 삭제**될 수 있음 |
| **개선 포인트** | 대여는 `rental` 테이블에서만 삭제, 고객은 `customer`에 유지(→ **2NF**) |

---

### Step 1 이후 체크리스트

- [ ] 삽입: 회원만 먼저 등록 가능한가?
- [ ] 갱신: 고객명·브랜드 요금 변경 시 **수정 row 수 = 1**인가?
- [ ] 삭제: 대여 삭제 시 고객·브랜드 마스터는 남는가?

아래 세 셀에서 각 이상을 **실행 가능한 SQL**로 확인합니다.

In [11]:
# 삽입 이상: 한 번도 대여를 하지 않은 신규 고객을 추가하려면?
# rental_time이 없으면 행을 구분할 수 없어 데이터가 의미 없어짐

print("삽입 이상 (Insert Anomaly)")
print("─" * 50)
print("시도: 대여 이력이 없는 신규 고객 'newuser' 추가")
print()

try:
    conn.execute("""
        INSERT INTO customer_unnormalized (customer_id, customer_name)
        VALUES ('newuser', '최신규')
    """)
    conn.commit()
    show("SELECT * FROM customer_unnormalized WHERE customer_id = 'newuser'")
    print("rental_time, brand 등 핵심 컬럼이 NULL인 채로 삽입됩니다.")
    print("    → 이 고객이 킥보드를 빌린 건지, 안 빌린 건지 알 수 없습니다.")
    # 데모 후 롤백
    conn.execute("DELETE FROM customer_unnormalized WHERE customer_id = 'newuser'")
    conn.commit()
except Exception as e:
    print(f"오류: {e}")

삽입 이상 (Insert Anomaly)
──────────────────────────────────────────────────
시도: 대여 이력이 없는 신규 고객 'newuser' 추가



,customer_id,customer_name,rental_time,rental_location,brand,model_year,price_per_min,basic_price,company
0,newuser,최신규,None,None,None,None,None,None,None


rental_time, brand 등 핵심 컬럼이 NULL인 채로 삽입됩니다.
    → 이 고객이 킥보드를 빌린 건지, 안 빌린 건지 알 수 없습니다.


In [12]:
# 갱신 이상: 'kmax6' 고객의 이름을 바꾸려면 모든 row를 갱신해야 함
# 일부 row만 수정하면 같은 고객인데 이름이 달라지는 불일치 발생

print("갱신 이상 (Update Anomaly)")
print("─" * 50)
print("시도: 'kmax6' 고객 이름을 '김민준' → '김민'으로 변경")
print("      (일부 row만 수정하는 실수 상황 시뮬레이션)")
print()

try:
    # 의도적으로 첫 번째 row만 수정
    conn.execute("""
        UPDATE customer_unnormalized
        SET customer_name = '김민'
        WHERE customer_id = 'kmax6'
        AND rental_time = '2020-08-20 13:01:02, 2020-09-01 20:39:52'
    """)
    conn.commit()

    show(
        "SELECT customer_id, customer_name, rental_time FROM customer_unnormalized WHERE customer_id = 'kmax6'",
        title="갱신 후 — 같은 고객인데 이름이 다름!",
    )

    # 원복
    conn.execute("UPDATE customer_unnormalized SET customer_name = '김민준' WHERE customer_id = 'kmax6'")
    conn.commit()
    print("중복 저장된 데이터 중 일부만 수정되면 불일치가 발생합니다.")
except Exception as e:
    print(f"오류: {e}")
    conn.rollback()

갱신 이상 (Update Anomaly)
──────────────────────────────────────────────────
시도: 'kmax6' 고객 이름을 '김민준' → '김민'으로 변경
      (일부 row만 수정하는 실수 상황 시뮬레이션)

📋 갱신 후 — 같은 고객인데 이름이 다름!


,customer_id,customer_name,rental_time
0,kmax6,김민,"2020-08-20 13:01:02, 2020-09-01 20:39:52"


중복 저장된 데이터 중 일부만 수정되면 불일치가 발생합니다.


In [13]:
# 삭제 이상: 'flykite' 고객의 대여 기록을 지우면 고객 정보도 함께 사라짐

print("삭제 이상 (Delete Anomaly)")
print("─" * 50)
print("시도: 'flykite(이서연)' 고객의 대여 기록 삭제")
print()

try:
    conn.execute("DELETE FROM customer_unnormalized WHERE customer_id = 'flykite'")
    conn.commit()

    show("SELECT * FROM customer_unnormalized", title="삭제 후 — 이서연 고객 정보가 완전히 사라짐!")
    print(" 대여 기록과 고객 정보가 같은 테이블에 있어서, 하나를 지우면 다른 것도 사라집니다.")

    # 원복
    conn.execute("""
        INSERT INTO customer_unnormalized VALUES
        ('flykite', '이서연', '2020-11-11 22:01:30',
         '서울시 동작구 대방동', 'willgo', 2018, 110, 950, 'everythere')
    """)
    conn.commit()
except Exception as e:
    print(f"오류: {e}")
    conn.rollback()

삭제 이상 (Delete Anomaly)
──────────────────────────────────────────────────
시도: 'flykite(이서연)' 고객의 대여 기록 삭제

📋 삭제 후 — 이서연 고객 정보가 완전히 사라짐!


,customer_id,customer_name,rental_time,rental_location,brand,model_year,price_per_min,basic_price,company
0,kmax6,김민준,"2020-08-20 13:01:02, 2020-09-01 20:39:52",서울시 강남구 역삼동,boardkick,2015,100,1000,elice
1,freeman123,박서준,2021-01-05 09:15:00,서울시 관악구 신림동,boardkick,2015,100,1000,elice


 대여 기록과 고객 정보가 같은 테이블에 있어서, 하나를 지우면 다른 것도 사라집니다.


## Step 2: 1차 정규화 (1NF)

**문제**: `rental_time` 컬럼에 두 개의 시간이 쉼표로 합쳐져 있음 → **원자값 위반**

**규칙**: 테이블의 모든 컬럼은 **하나의 값(원자값)** 만 가져야 합니다.

**해결**: 하나의 컬럼에 하나의 값만 저장하고, 대여 시간이 여러 개면 row를 분리합니다.

In [14]:
# Step 4(3NF) 테이블이 남아 있으면 FK 때문에 DROP 실패 → 먼저 하위 테이블 삭제
drop_from_1nf()

conn.executescript("""
CREATE TABLE customer_1nf (
    customer_id      TEXT,
    customer_name    TEXT,
    rental_time      TEXT,   -- 이제 하나의 값만 저장
    rental_location  TEXT,
    brand            TEXT,
    model_year       INTEGER,
    price_per_min    INTEGER,
    basic_price      INTEGER,
    company          TEXT
);
""")

# rental_time에 쉼표가 있으면 분리해서 삽입
cursor = conn.execute("SELECT * FROM customer_unnormalized")
rows = cursor.fetchall()

for row in rows:
    cid, name, times, loc, brand, year, ppm, bp, comp = row
    for t in [t.strip() for t in times.split(",")]:
        conn.execute(
            """
            INSERT INTO customer_1nf VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (cid, name, t, loc, brand, year, ppm, bp, comp),
        )

conn.commit()

print("Before (비정규화):")
show("SELECT customer_id, customer_name, rental_time FROM customer_unnormalized")

print("\nAfter (1NF 적용):")
show("SELECT customer_id, customer_name, rental_time FROM customer_1nf")
print("\n 1NF 완료: 모든 컬럼이 원자값을 가집니다.")

Before (비정규화):


,customer_id,customer_name,rental_time
0,kmax6,김민준,"2020-08-20 13:01:02, 2020-09-01 20:39:52"
1,freeman123,박서준,2021-01-05 09:15:00
2,flykite,이서연,2020-11-11 22:01:30



After (1NF 적용):


,customer_id,customer_name,rental_time
0,kmax6,김민준,2020-08-20 13:01:02
1,kmax6,김민준,2020-09-01 20:39:52
2,freeman123,박서준,2021-01-05 09:15:00
3,flykite,이서연,2020-11-11 22:01:30



 1NF 완료: 모든 컬럼이 원자값을 가집니다.


## Step 3: 2차 정규화 (2NF)

**문제**: 기본키는 `(customer_id, rental_time)` 복합키인데,  
`customer_name`은 `customer_id` 하나만으로 결정됨 → **부분 함수 종속**

```
(customer_id, rental_time) → rental_location, brand ...  완전 함수 종속
customer_id → customer_name                               부분 함수 종속
```

**해결**: `customer_name`을 별도 테이블로 분리합니다.

In [15]:
# rental_3nf가 customer_2nf를 참조 → 3NF 테이블을 먼저 삭제 (customer_1nf는 유지)
drop_from_2nf()

conn.executescript("""
-- 고객 정보 테이블 (customer_id만으로 결정되는 속성)
CREATE TABLE customer_2nf (
    customer_id   TEXT PRIMARY KEY,
    customer_name TEXT NOT NULL
);

-- 대여 정보 테이블 (복합키로 결정되는 속성)
CREATE TABLE rental_2nf (
    customer_id      TEXT,
    rental_time      TEXT,
    rental_location  TEXT,
    brand            TEXT,
    model_year       INTEGER,
    price_per_min    INTEGER,
    basic_price      INTEGER,
    company          TEXT,
    PRIMARY KEY (customer_id, rental_time),
    FOREIGN KEY (customer_id) REFERENCES customer_2nf(customer_id)
);
""")

# 데이터 이전
conn.execute("""
    INSERT INTO customer_2nf
    SELECT DISTINCT customer_id, customer_name FROM customer_1nf
""")
conn.execute("""
    INSERT INTO rental_2nf
    SELECT customer_id, rental_time, rental_location, brand,
           model_year, price_per_min, basic_price, company
    FROM customer_1nf
""")
conn.commit()

show("SELECT * FROM customer_2nf", title="customer 테이블 (분리됨)")
show("SELECT * FROM rental_2nf", title="rental 테이블")
print("\n2NF 완료: 부분 함수 종속을 제거했습니다.")
print("   → 이제 대여 이력 없이도 고객을 등록할 수 있습니다!")

📋 customer 테이블 (분리됨)


,customer_id,customer_name
0,kmax6,김민준
1,freeman123,박서준
2,flykite,이서연


📋 rental 테이블


,customer_id,rental_time,rental_location,brand,model_year,price_per_min,basic_price,company
0,kmax6,2020-08-20 13:01:02,서울시 강남구 역삼동,boardkick,2015,100,1000,elice
1,kmax6,2020-09-01 20:39:52,서울시 강남구 역삼동,boardkick,2015,100,1000,elice
2,freeman123,2021-01-05 09:15:00,서울시 관악구 신림동,boardkick,2015,100,1000,elice
3,flykite,2020-11-11 22:01:30,서울시 동작구 대방동,willgo,2018,110,950,everythere



2NF 완료: 부분 함수 종속을 제거했습니다.
   → 이제 대여 이력 없이도 고객을 등록할 수 있습니다!


## Step 4: 3차 정규화 (3NF)

**문제**: `rental_2nf` 테이블에서 이행 함수 종속이 존재합니다.

```
(customer_id, rental_time) → brand → company    이행 함수 종속
brand → price_per_min                            이행 함수 종속
```

기본키가 아닌 `brand`가 `company`, `price_per_min`을 결정하고 있습니다.

**해결**: `brand` 관련 정보를 별도 테이블로 분리합니다.

In [16]:
conn.executescript("""
DROP TABLE IF EXISTS rental_3nf;
DROP TABLE IF EXISTS kickboard_3nf;

-- 킥보드/브랜드 정보 테이블
CREATE TABLE kickboard_3nf (
    brand         TEXT PRIMARY KEY,
    model_year    INTEGER,
    price_per_min INTEGER,
    basic_price   INTEGER,
    company       TEXT
);

-- 대여 정보 테이블 (이행 종속 제거)
CREATE TABLE rental_3nf (
    customer_id      TEXT,
    rental_time      TEXT,
    rental_location  TEXT,
    brand            TEXT,
    PRIMARY KEY (customer_id, rental_time),
    FOREIGN KEY (customer_id) REFERENCES customer_2nf(customer_id),
    FOREIGN KEY (brand)       REFERENCES kickboard_3nf(brand)
);
""")

conn.execute("""
    INSERT INTO kickboard_3nf
    SELECT DISTINCT brand, model_year, price_per_min, basic_price, company
    FROM rental_2nf
""")
conn.execute("""
    INSERT INTO rental_3nf
    SELECT customer_id, rental_time, rental_location, brand
    FROM rental_2nf
""")
conn.commit()

show("SELECT * FROM rental_3nf", title="rental 테이블 (이행 종속 제거)")
show("SELECT * FROM kickboard_3nf", title="kickboard 테이블 (분리됨)")
print("\n3NF 완료: 이행 함수 종속을 제거했습니다.")

📋 rental 테이블 (이행 종속 제거)


,customer_id,rental_time,rental_location,brand
0,kmax6,2020-08-20 13:01:02,서울시 강남구 역삼동,boardkick
1,kmax6,2020-09-01 20:39:52,서울시 강남구 역삼동,boardkick
2,freeman123,2021-01-05 09:15:00,서울시 관악구 신림동,boardkick
3,flykite,2020-11-11 22:01:30,서울시 동작구 대방동,willgo


📋 kickboard 테이블 (분리됨)


,brand,model_year,price_per_min,basic_price,company
0,boardkick,2015,100,1000,elice
1,willgo,2018,110,950,everythere



3NF 완료: 이행 함수 종속을 제거했습니다.


## Step 5: 최종 결과 검증

정규화 후에도 JOIN을 통해 원본 데이터를 동일하게 조회할 수 있습니다.

In [17]:
show("""
    SELECT
        c.customer_id,
        c.customer_name,
        r.rental_time,
        r.rental_location,
        r.brand,
        k.model_year,
        k.price_per_min,
        k.basic_price,
        k.company
    FROM rental_3nf r
    JOIN customer_2nf  c ON r.customer_id = c.customer_id
    JOIN kickboard_3nf k ON r.brand       = k.brand
    ORDER BY r.rental_time
""", title="최종 JOIN 결과 — 원본 데이터와 동일!")

print("\n🎉 정규화 완료!")
print("   이상 현상 없이 동일한 데이터를 조회할 수 있습니다.")

📋 최종 JOIN 결과 — 원본 데이터와 동일!


,customer_id,customer_name,rental_time,rental_location,brand,model_year,price_per_min,basic_price,company
0,kmax6,김민준,2020-08-20 13:01:02,서울시 강남구 역삼동,boardkick,2015,100,1000,elice
1,kmax6,김민준,2020-09-01 20:39:52,서울시 강남구 역삼동,boardkick,2015,100,1000,elice
2,flykite,이서연,2020-11-11 22:01:30,서울시 동작구 대방동,willgo,2018,110,950,everythere
3,freeman123,박서준,2021-01-05 09:15:00,서울시 관악구 신림동,boardkick,2015,100,1000,elice



🎉 정규화 완료!
   이상 현상 없이 동일한 데이터를 조회할 수 있습니다.


In [18]:
# 정규화 전후 비교 요약
summary = {
    "단계": ["비정규화", "1NF", "2NF", "3NF"],
    "테이블 수": [1, 1, 2, 3],
    "제거된 문제": [
        "없음",
        "다중값 컬럼 제거 (원자값 보장)",
        "부분 함수 종속 제거",
        "이행 함수 종속 제거",
    ],
    "남은 이상 현상": [
        "삽입/갱신/삭제 이상 모두 존재",
        "갱신/삭제 이상 여전히 존재",
        "일부 해소, 이행 종속 남음",
        "없음 ",
    ],
}

display(pd.DataFrame(summary))

,단계,테이블 수,제거된 문제,남은 이상 현상
0,비정규화,1,없음,삽입/갱신/삭제 이상 모두 존재
1,1NF,1,다중값 컬럼 제거 (원자값 보장),갱신/삭제 이상 여전히 존재
2,2NF,2,부분 함수 종속 제거,"일부 해소, 이행 종속 남음"
3,3NF,3,이행 함수 종속 제거,없음
